In [17]:
from typing import Dict, List, TypedDict
from langgraph.graph import StateGraph, END
import random

In [18]:
class AgentState(TypedDict):
    player_name: str
    guess: List[int]
    attempts: int
    lower_bound: int
    higher_bound: int
    answer: int

In [22]:
def setup_node(state: AgentState) -> AgentState:
    """ This node is to set up everything! """

    state['attempts'] =  0
    state['answer'] = random.randint(1, 20)
    state['guess'] = []
 
    return state

def guess(state: AgentState) -> AgentState:
    """ This is a random node """
    
    possible_guesses = [i for i in range(state['lower_bound'], state['higher_bound']+1) if i not in state['guess']]
    if possible_guesses:
        guess = random.choice(possible_guesses)
    else:
        guess = random.randint(state['lower_bound'], state['higher_bound'])

    state['guess'].append(guess)
    state['attempts'] += 1

    return state

def hint(state: AgentState) -> AgentState:
    """ This is hint node would say higher or lower """

    latest_guess = state['guess'][-1]
    answer = state['answer']

    if latest_guess < answer:
        state['lower_bound'] = max(state['lower_bound'], latest_guess+1)
        
    elif latest_guess > answer:
        state['higher_bound'] = min(state['higher_bound'], latest_guess-1)

    return state

def if_continue(state: AgentState) -> str:
    """ This is hint node would say higher or lower """

    latest_guess = state['guess'][-1]

    if latest_guess == state['answer']:
        print('success')
        return "exit"
        
    elif state['attempts'] < 7:
        return "loop"
    else:
        return "exit"


In [23]:
graph = StateGraph(AgentState)

graph.add_node("setup_node", setup_node)
graph.add_node("random", guess)
graph.add_node("hint", hint)

graph.add_conditional_edges(

    "hint",
    if_continue,

    {
        "loop": "random",
        "exit": END
    }
)

graph.set_entry_point("setup_node")
graph.add_edge("setup_node", "random")
graph.add_edge("random", "hint")

app = graph.compile()

In [24]:
result = app.invoke({"player_name": "Student", "guess": [], "attempts": 0, "lower_bound": 1, "higher_bound": 20})

success


In [ ]:
print(app.invoke(initial_state))

{'num1': 10, 'num2': 5, 'num3': 7, 'num4': 2, 'operation1': '-', 'operation2': '+', 'fnum1': 5, 'fnum2': 9}
